In [ ]:
from pathlib import Path
import re
import pandas as pd


# Find repo root 
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").is_dir() and ((p / ".git").exists() or (p / "README.md").exists() or (p / "src").is_dir()):
            return p
    for p in [start] + list(start.parents):
        if (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root. Open the notebook from inside the repo folder.")

REPO_ROOT = find_repo_root(Path.cwd())


def find_first_under_data(filename: str) -> Path:
    data_dir = REPO_ROOT / "data"
    hits = sorted([p for p in data_dir.rglob(filename) if p.is_file()])
    if not hits:
        raise FileNotFoundError(f"Could not find {filename} under {data_dir}")
    return hits[0]


# Locate inputs automatically
PII_PATH = find_first_under_data("pii_inventory.csv")
ANALYSIS_PATH = find_first_under_data("applications_analysis.csv")


OUT_DIR = REPO_ROOT / "data" / "governance" / "pii_fields_study"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# Load
pii = pd.read_csv(PII_PATH)
analysis = pd.read_csv(ANALYSIS_PATH)

print("Loaded pii_inventory rows =", len(pii))
print("Loaded analysis shape =", analysis.shape)


# Presence matrix + direct PII list 
# Expect columns: field_path, classification, present_in_raw, present_in_curated, present_in_analysis
presence_matrix = (
    pii.rename(columns={"field_path": "field_name", "classification": "pii_class"})
      .loc[:, ["field_name", "pii_class", "present_in_raw", "present_in_curated", "present_in_analysis"]]
      .sort_values(["pii_class", "field_name"])
)
presence_matrix.to_csv(OUT_DIR / "pii_presence_matrix.csv", index=False)

direct_pii_fields = (
    pii.loc[pii["classification"].astype(str).str.strip().str.lower() == "pii", "field_path"]
      .dropna().astype(str).str.strip().sort_values().unique().tolist()
)
(Path(OUT_DIR / "direct_pii_fields_list.txt")).write_text("\n".join(direct_pii_fields), encoding="utf-8")


# Structural checks 
direct_set = set(direct_pii_fields)
exact_present = [c for c in analysis.columns if c in direct_set]

direct_leaf = {f.split(".")[-1].lower() for f in direct_pii_fields}
leaf_present = [c for c in analysis.columns if c.lower() in direct_leaf]

print("Direct PII columns found (exact match):", exact_present)
print("Direct PII columns found (leaf-name match):", leaf_present)


# Full leakage scan on text columns only
scan_df = analysis.select_dtypes(include=["object"]).fillna("").astype(str)

patterns = {
    "email_like": re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    "ipv4_like": re.compile(r"\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b"),
    "ssn_like_hyphen": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
}

summary_rows = []
hit_details = []

if scan_df.shape[1] == 0:
    print("No text columns to scan.")
else:
    for name, pat in patterns.items():
        hits_bool = scan_df.apply(lambda col: col.str.contains(pat, regex=True, na=False))
        per_col_hits = hits_bool.sum(axis=0)

        cols_with_hits = per_col_hits[per_col_hits > 0].sort_values(ascending=False)
        cell_hits = int(per_col_hits.sum())

        summary_rows.append({
            "pattern": name,
            "sample_size_rows": int(scan_df.shape[0]),
            "columns_scanned": int(scan_df.shape[1]),
            "cell_hits": cell_hits,
            "columns_with_hits": int((per_col_hits > 0).sum()),
        })

        for col_name, n_hits in cols_with_hits.items():
            hit_details.append({"pattern": name, "column": col_name, "hits": int(n_hits)})

leak_summary = pd.DataFrame(summary_rows, columns=["pattern","sample_size_rows","columns_scanned","cell_hits","columns_with_hits"])
hit_details_df = pd.DataFrame(hit_details, columns=["pattern","column","hits"])

leak_summary.to_csv(OUT_DIR / "analysis_pii_leakage_scan_summary.csv", index=False)
hit_details_df.to_csv(OUT_DIR / "analysis_pii_leakage_by_column.csv", index=False)



print("\nLeakage scan summary:")
print(leak_summary if len(leak_summary) else "(no text columns scanned)")

if hit_details_df.shape[0] == 0:
    print("\nNo leakage hits by column (expected for a PII-safe analysis extract).")
else:
    print("\nLeakage hits by column (investigate these columns):")
    print(hit_details_df.sort_values(["pattern","hits"], ascending=[True, False]).head(50))

Loaded pii_inventory rows = 10
Loaded analysis shape = (500, 16)
Direct PII columns found (exact match): []
Direct PII columns found (leaf-name match): []

Leakage scan summary:
           pattern  sample_size_rows  columns_scanned  cell_hits  \
0       email_like               500                6          0   
1        ipv4_like               500                6          0   
2  ssn_like_hyphen               500                6          0   

   columns_with_hits  
0                  0  
1                  0  
2                  0  

No leakage hits by column (expected for a PII-safe analysis extract).
